# DeepGuard v2 · Official FF++ copy (Drive) → private Kaggle datasets

Creates four **private** Kaggle datasets from `My Drive/DeepGuard/raw/ffpp/`, keeping the official folder layout:

| Kaggle dataset | Contents | Size |
|---|---|---|
| `dg-ffpp-c40` | FF++ originals + 5 methods, c40 | ≈ 1.5 GB |
| `dg-ffpp-c23` | FF++ originals + 5 methods, c23 | ≈ 11 GB |
| `dg-dfd-c40` | DeepFakeDetection originals + fakes, c40 | ≈ 3.4 GB |
| `dg-dfd-c23` | DeepFakeDetection originals + fakes, c23 | ≈ 24 GB |

**First run uploads only `dg-ffpp-c40`** (≈ 5 min) as a test. Check it on Kaggle, then set `UPLOAD_ONLY = None` in Cell 3 and run Cells 3–4 again for the other three.

**Before running**
- Runtime → Change runtime type → **CPU**.
- Colab secret **`KAGGLE_API_TOKEN`** (Kaggle → Settings → API Tokens → *Generate New Token*), with *Notebook access* on.
- The download notebook must be finished, including its Cell 6 (`MANIFEST.csv`).
- Re-running is safe: datasets that already exist on Kaggle are skipped.

In [ ]:
# ===== CELL 1: Setup — Drive, Kaggle CLI, login check =====
import os, sys, json, time, shutil, zipfile, subprocess
from pathlib import Path
import pandas as pd

T0 = time.time()
def log(msg):
    """Print with elapsed time so progress is visible."""
    m, s = divmod(int(time.time() - T0), 60)
    print(f"[{m:3d}m{s:02d}s] {msg}", flush=True)

LOCAL_TEST = os.environ.get("DG_LOCAL_TEST") == "1"     # only for offline testing of this notebook
if not LOCAL_TEST:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle>=1.8.0"], check=True)
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN").strip()   # never printed
DRIVE = Path(os.environ.get("DG_DRIVE", "/content/drive/MyDrive"))
RAW = DRIVE / "DeepGuard" / "raw" / "ffpp"
STAGE = Path(os.environ.get("DG_STAGE", "/content/stage"))            # local Colab disk, temporary zips
UP_STATUS = RAW / "_kaggle_upload_status.json"
OWNER = "mynkey"

def kaggle(*args, timeout=None):
    """Run the Kaggle CLI and return (exit code, output text)."""
    r = subprocess.run(["kaggle", *args], capture_output=True, text=True, timeout=timeout)
    return r.returncode, (r.stdout + r.stderr).strip()

rc, out = kaggle("--version"); log(f"Kaggle CLI: {out}")
rc, out = kaggle("datasets", "list", "--mine")
assert rc == 0, f"Kaggle login failed — check the KAGGLE_API_TOKEN secret.\n{out[-500:]}"
log(f"Kaggle login OK (your datasets listed: {max(0, len(out.splitlines()) - 2)})")

In [ ]:
# ===== CELL 2: Plan — which folders go into which dataset =====
SETS = {
    "dg-ffpp-c40": ("FF++ official c40 (DeepGuard)", "c40",
                    ["original_sequences/youtube"] + [f"manipulated_sequences/{m}" for m in
                     ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures", "FaceShifter"]]),
    "dg-ffpp-c23": ("FF++ official c23 (DeepGuard)", "c23",
                    ["original_sequences/youtube"] + [f"manipulated_sequences/{m}" for m in
                     ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures", "FaceShifter"]]),
    "dg-dfd-c40": ("DFD official c40 (DeepGuard)", "c40",
                   ["original_sequences/actors", "manipulated_sequences/DeepFakeDetection"]),
    "dg-dfd-c23": ("DFD official c23 (DeepGuard)", "c23",
                   ["original_sequences/actors", "manipulated_sequences/DeepFakeDetection"]),
}
for slug, (title, _, _) in SETS.items():
    assert 6 <= len(title) <= 50, f"Kaggle titles must be 6–50 characters: {title!r} has {len(title)}"
assert (RAW / "MANIFEST.csv").exists(), "Run Cell 6 of the download notebook first (it writes MANIFEST.csv)."
man = pd.read_csv(RAW / "MANIFEST.csv")
plan = {}
for slug, (title, comp, folders) in SETS.items():
    files = []
    for f in folders:
        files += sorted((RAW / f / comp / "videos").glob("*.mp4"))
    rels = {str(p.relative_to(RAW)) for p in files}
    in_manifest = man.rel.isin(rels).sum()
    gb = sum(p.stat().st_size for p in files) / 1e9
    plan[slug] = files
    print(f"  {slug:12s} {len(files):5d} files  {gb:6.2f} GB   in MANIFEST: {in_manifest}/{len(files)}")
    assert in_manifest == len(files), f"{slug}: some files are not in MANIFEST.csv — re-run Cell 6 of the download notebook"
free = shutil.disk_usage("/content" if Path("/content").exists() else "/").free / 1e9
log(f"Free local disk for temporary zips: {free:.0f} GB (largest dataset needs ≈ 25 GB)")
up_status = json.load(open(UP_STATUS)) if UP_STATUS.exists() else {}

In [ ]:
# ===== CELL 3: Upload — zip each dataset (no compression: videos are already compressed) and create it on Kaggle =====
# Inside each zip the official layout is kept, e.g. original_sequences/youtube/c23/videos/000.mp4
# Kaggle unpacks the zip, so notebooks see /kaggle/input/<dataset>/<zip name>/original_sequences/...
def build_zip(slug, files):
    out_dir = STAGE / slug
    zpath = out_dir / f"{slug.replace('-', '_')}.zip"
    if zpath.exists():                                     # a complete zip left by a failed upload → reuse it
        try:
            with zipfile.ZipFile(zpath) as z:
                if len(z.namelist()) == len(files):
                    log(f"   reusing the complete zip from the earlier attempt ({zpath.stat().st_size/1e9:.2f} GB)")
                    return out_dir, zpath
        except zipfile.BadZipFile:
            pass
    shutil.rmtree(out_dir, ignore_errors=True); out_dir.mkdir(parents=True)
    need = sum(p.stat().st_size for p in files) / 1e9
    free = shutil.disk_usage(out_dir).free / 1e9
    assert free > need * 1.05 + 2, f"Not enough local disk: need {need:.1f} GB, free {free:.1f} GB"
    t, done = time.time(), 0
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_STORED, allowZip64=True) as z:
        for k, p in enumerate(files, 1):
            z.write(p, str(p.relative_to(RAW)))
            done += p.stat().st_size
            if k % 500 == 0 or k == len(files):
                log(f"   zipped {k}/{len(files)} files ({done/1e9:.2f} GB, {done/1e6/max(time.time()-t,1):.0f} MB/s)")
    with zipfile.ZipFile(zpath) as z:                      # quick self-check of the zip
        assert len(z.namelist()) == len(files), "zip is missing files"
    return out_dir, zpath

UPLOAD_ONLY = ["dg-ffpp-c40"]   # FIRST run: only the smallest (1.5 GB) as a test. After you have checked it on Kaggle, set to None for the rest.

for slug, (title, comp, folders) in SETS.items():
    if UPLOAD_ONLY and slug not in UPLOAD_ONLY:
        continue
    ref = f"{OWNER}/{slug}"
    rc, out = kaggle("datasets", "status", ref)
    if rc == 0 and "ready" in out.lower():
        log(f"{slug}: already on Kaggle (status: {out}) → skipped"); up_status[slug] = "ready"; continue
    if up_status.get(slug) == "uploaded":
        log(f"{slug}: uploaded earlier, Kaggle still processing (status: {out}) → skipped"); continue
    log(f"{slug}: building zip …")
    out_dir, zpath = build_zip(slug, plan[slug])
    meta = {"title": title, "id": ref, "licenses": [{"name": "other"}],
            "subtitle": "Official FaceForensics++ download. Private; not for redistribution.",
            "description": ("Official FaceForensics++ / DeepFakeDetection videos downloaded with the authors' script "
                            "(access approved 15 Nov 2026). Used under the FaceForensics Terms of Use for the "
                            "non-commercial DeepGuard project. PRIVATE — do not share or redistribute. "
                            "Checksums: MANIFEST.csv on Google Drive (DeepGuard/raw/ffpp).")}
    json.dump(meta, open(out_dir / "dataset-metadata.json", "w"), indent=1)
    log(f"{slug}: uploading {zpath.stat().st_size/1e9:.2f} GB to Kaggle (private) …")
    t = time.time()
    rc, out = kaggle("datasets", "create", "-p", str(out_dir), "-q")
    (RAW / f"_kaggle_log_{slug}.txt").write_text(out)
    log(f"{slug}: create exit code {rc} after {(time.time()-t)/60:.1f} min")
    print("   " + "\n   ".join(out.splitlines()[-6:]))
    if rc != 0:
        print(f"   STOPPED: upload of {slug} failed. Send Claude the lines above.")
        break
    up_status[slug] = "uploaded"
    json.dump(up_status, open(UP_STATUS, "w"), indent=1)
    shutil.rmtree(out_dir, ignore_errors=True)            # free the local disk for the next zip
log(f"Upload status: {up_status}")

In [ ]:
# ===== CELL 4: Wait until Kaggle has finished processing (unzipping) every dataset =====
POLL = int(os.environ.get("DG_POLL", "60"))
pending = [s for s in SETS if up_status.get(s) in ("uploaded", "ready")]
deadline = time.time() + 3 * 3600
while pending and time.time() < deadline:
    still = []
    for slug in pending:
        rc, out = kaggle("datasets", "status", f"{OWNER}/{slug}")
        state = out.strip().splitlines()[-1] if out else "?"
        if rc == 0 and "ready" in state.lower():
            if up_status.get(slug) != "ready":
                log(f"{slug}: READY")
            up_status[slug] = "ready"
        else:
            still.append(slug); print(f"      {slug}: {state}", flush=True)
    json.dump(up_status, open(UP_STATUS, "w"), indent=1)
    pending = still
    if pending:
        time.sleep(POLL)
for slug in SETS:
    print(f"  {slug:12s} {up_status.get(slug, 'not uploaded'):12s} https://www.kaggle.com/datasets/{OWNER}/{slug}")
log("Next: attach all four to the audit notebook (Add Input → Your Datasets).")